In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
import matplotlib.pyplot as plt
from openai import OpenAI
import random
import pickle
import json

In [2]:
load_dotenv(override=True)
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)
openai = OpenAI()


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from items import Item
from testing import Tester

In [4]:
%matplotlib inline

In [5]:
with open('train.pkl', 'rb') as file:
    train = pickle.load(file=file)

with open('test.pkl', 'rb') as file:
    test = pickle.load(file=file)

In [6]:
train_fine_tune = train[:500]
validation_fine_tune = train[500:550]

In [7]:
wandb_integration = {"type": "wandb", "wandb": {"project": "gpt-pricer"}}

In [8]:
def messages_for(item):
    system_message = "You estimate prices of items. Reply only with the price, no explanation"
    user_prompt = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"}
    ]

In [9]:
def make_jsonl(items):
    result = ""
    for item in items:
        message = messages_for(item)
        message_str = json.dumps(message)
        result += '{"messages": ' +  message_str + '}\n'
    return result.strip()

In [10]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "system", "content": "You estimate prices of items. Reply only with the price, no explanation"}, {"role": "user", "content": "How much does this cost?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7"}, {"role": "assistant", "content": "Price is $226.95"}]}
{"messages": [{"role": "system", "content": "You estimate prices of items. 

In [11]:
def write_jsonl(items, filename):
    with open(filename, 'w') as f:
        jsonl_file = make_jsonl(items)
        f.write(jsonl_file)

In [ ]:
write_jsonl(train_fine_tune, 'fine_tune_train.jsonl')

In [13]:
write_jsonl(validation_fine_tune, 'fine_tune_validation.jsonl')

In [14]:
with open('fine_tune_train.jsonl', 'rb') as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [15]:
with open('fine_tune_validation.jsonl', 'rb') as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [16]:
validation_file

FileObject(id='file-EGDho84UCrZiJbnEhvLQ2M', bytes=47059, created_at=1762599160, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [18]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model='gpt-4.1-mini-2025-04-14',
    seed=42,
    hyperparameters={"n_epochs": 1},
    integrations=[wandb_integration],
    suffix='pricer',
)

FineTuningJob(id='ftjob-CadpAbwwVhWUU8C02FYHJr14', created_at=1762599428, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-PwbEgl7hLswmei0amEMrqbLi', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-WbMEr1Hq9ECY8Lv3riRh3i', validation_file='file-EGDho84UCrZiJbnEhvLQ2M', estimated_finish=None, integrations=[FineTuningJobWandbIntegrationObject(type='wandb', wandb=FineTuningJobWandbIntegration(project='gpt-pricer', entity=None, name=None, tags=None, run_id='ftjob-CadpAbwwVhWUU8C02FYHJr14'))], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pric

In [25]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [26]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-CadpAbwwVhWUU8C02FYHJr14', created_at=1762599428, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=1), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-PwbEgl7hLswmei0amEMrqbLi', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-WbMEr1Hq9ECY8Lv3riRh3i', validation_file='file-EGDho84UCrZiJbnEhvLQ2M', estimated_finish=None, integrations=[FineTuningJobWandbIntegrationObject(type='wandb', wandb=FineTuningJobWandbIntegration(project='gpt-pricer', entity=None, name=None, tags=None, run_id='ftjob-CadpAbwwVhWUU8C02FYHJr14'))], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=1))), user_provided_suffix='pricer', usage_metri

In [31]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=20).data

[FineTuningJobEvent(id='ftevent-J87EKbPkK3RC5g0eLRJx7uMi', created_at=1762600351, level='info', message='Evaluating model against our usage policies', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-eUMe1KaXfMbmVgyeDg8gsBwB', created_at=1762600351, level='info', message='New fine-tuned model created', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-s4HOhrkjdptWyG7p8u94qRmn', created_at=1762600330, level='info', message='Step 500/500: training loss=1.13, validation loss=1.63, full validation loss=1.12', object='fine_tuning.job.event', data={'step': 500, 'train_loss': 1.1333951950073242, 'valid_loss': 1.628415822982788, 'total_steps': 500, 'full_valid_loss': 1.1157926368713378, 'train_mean_token_accuracy': 0.75, 'valid_mean_token_accuracy': 0.75, 'full_valid_mean_token_accuracy': 0.7975}, type='metrics'),
 FineTuningJobEvent(id='ftevent-YxcSADFNlApU8w1w03ZmfbM4', created_at=1762600320, level='info', messa